# 3.1 - Data Cleaning


![cleaning](images/cleaning.jpg)


$$$$

Una vez que hemos conseguido los datos necesarios, es necesario observar su calidad y orden. No se trata de un an?lisis exploratorio de los datos, sino de ver como de sucios y desordenados est?n nuestros datos. Cuando los datos tienen problemas de calidad decimos que los datos est?n sucios. Los problemas de calidad est?n relacionados con valores nulos, datos inconsistentes, tipo de dato incorrecto y registros duplicados. 


$$$$
![assess](images/data_assess.png)
$$$$
$$$$

Por otro lado, el orden de los datos est? relacionado con su estructura. Como dice [Hadley Wickham](http://hadley.nz/) en su paper [Tidy Data](https://cran.r-project.org/web/packages/tidyr/vignettes/tidy-data.html), los datos est?n ordenados cuando:

+ Cada variable forma una columna.
+ Cada observaci?n forma una fila.
+ Cada tipo de unidad observacional forma una tabla.
$$$$
![clean](images/data_clean.png)




# Ejemplo...

### Importaciones y carga de datos.

In [ ]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)  # me muestre todas las columnas
#pd.set_option('display.max_rows', None)  # me muestre todas las filas

In [ ]:
df = pd.read_csv('../data/vehicles_messy.csv')

df.head()

### Exploraci?n

In [ ]:
dim_original = df.shape

df.shape

In [ ]:
df.columns

In [ ]:
df.make.value_counts()

In [ ]:
df.dtypes

In [ ]:
df.info(memory_usage='deep')

In [ ]:
# columnas con valores nulos

nan_cols = df.isna().sum()

nan_cols[nan_cols>0]

In [ ]:
nan_cols[nan_cols>1e4].index

### Borrado de columnas con m?s de 10.000 valores nulos

In [ ]:
# df = df.drop(columns = nan_cols[nan_cols>1e4].index)

df.drop(columns = nan_cols[nan_cols>1e4].index, inplace=True)

In [ ]:
nan_cols = df.isna().sum()

nan_cols[nan_cols>0]

In [ ]:
dim_original, df.shape

### Exploraci?n columnas `cylinders` y `displ`

In [ ]:
# indices donde cylinders es nulo


bad_index = df.cylinders[df.cylinders.isna()].index

bad_index[:10]

In [ ]:
len(bad_index)

In [ ]:
df.iloc[bad_index][['make', 'model', 'fuelType', 'cylinders', 'displ']].fuelType.unique()

**Espera, son el?ctricos, de hecho existe un Mazda con motor rotatorio. Efectivamente no tienen cilindros ni cilindrada**

In [ ]:
# rellena con cero los valores nulos en esas 2 columnas

df[['cylinders', 'displ']] = df[['cylinders', 'displ']].fillna(0)

In [ ]:
nan_cols = df.isna().sum()

nan_cols[nan_cols>0]

### Exploraci?n columna `drive`

In [ ]:
# indices donde drive es nulo

bad_index = df.drive[df.drive.isna()].index

bad_index[:10]

In [ ]:
df.iloc[bad_index][['make', 'model', 'fuelType', 'drive']].fuelType.unique()

In [ ]:
df.drive.unique()

In [ ]:
df.iloc[bad_index].model.unique()

**Este es el punto donde siempre hay que decidir. ?Qu? nos conviene m?s, borrar registros o rellenar con un `unknown`? ?o usar chatGPT?**

In [ ]:
df.drive.fillna('unknown', inplace=True)

In [ ]:
nan_cols = df.isna().sum()

nan_cols[nan_cols>0]

In [ ]:
df[df.drive=='unknown'].head()

### Exploraci?n columna `trany`

In [ ]:
df.trany.head()

In [ ]:
df.trany[df.trany.isna()]

In [ ]:
df.trany.fillna('unknown', inplace=True)

In [ ]:
nan_cols = df.isna().sum()

nan_cols[nan_cols>0]

**Hasta aqu? la limpieza de nulos.**

### Eliminaci?n de duplicados

In [ ]:
df.duplicated().any()  # any = ?hay alguno True?

In [ ]:
df.drop_duplicates().shape == df.shape    # NO hay duplicados

In [ ]:
df[df.duplicated()]

In [ ]:
# ambas lineas dicen los mismo, elimina filas duplicadas

df = df.drop_duplicates()

df.drop_duplicates(inplace=True)

In [ ]:
df.head()

### Columnas constantes o de baja varianza

Nos fijaremos en las columnas num?ricas.

In [ ]:
type(np.nan)

In [ ]:
# columnas constantes

cte_cols = []


for c in df.select_dtypes(include=np.number):  # para cada columna numerica...
    
    if len(df[c].unique()) == 1:
        cte_cols.append(c)
        
cte_cols

In [ ]:
df['charge120'].unique()

In [ ]:
df.drop(columns = ['charge120'], inplace=True)

In [ ]:
# columnas constantes

cte_cols = []


for c in df.select_dtypes(include=np.number):  # para cada columna numerica...
    
    if df[c].std() == 0:
        cte_cols.append(c)
        
cte_cols

In [ ]:
# columnas ctes

cte_str_cols = []

for c in df.select_dtypes(include='object'):   # para cada columna NO numerica...
    
    if len(df[c].unique())==1:
        cte_str_cols.append(c)
        
        
cte_str_cols

In [ ]:
df.shape

In [ ]:
df.describe().T

In [ ]:
# cols baja var

low_var = []

for c in df.select_dtypes(include=np.number):  # para cada columna numerica...
    
    minimo = df[c].min()
    maximo = df[c].max()
    
    per_90 = np.percentile(df[c], 90)
    per_10 = np.percentile(df[c], 10)
    
    
    if minimo==per_90 or maximo==per_10:
        low_var.append(c)
        
low_var

In [ ]:
df.drop(columns = low_var, inplace=True)

In [ ]:
dim_original, df.shape

### Outliers (datos at?picos)

Outlier se refiere a esos registros con valores extremos, fuera del rango intercuartil. Dichos valores podr?an ser datos verdaderos o fallos de registro. Sea como fuere, se suelen eliminar estos registros porque podr?an desplazar la media de una manera ficticia. Esto es, la robustez de los datos. Los datos at?picos 'pesan m?s' que los datos cercanos a la media. Un solo valor es suficiente para influenciar enormemente la media del conjunto de datos. Esta vez nos fijaremos en los datos por filas.

In [ ]:
stats = df.describe().T

stats['IQR'] = stats['75%'] - stats['25%']

stats

In [ ]:
%pip install matplotlib

In [ ]:
%matplotlib inline

In [ ]:
df[['youSaveSpend']].boxplot(figsize=(15, 8));

In [ ]:
df.boxplot(figsize=(15, 8));

### Colinealidad  

Columnas con la misma informaci?n.

In [ ]:
df.corr()

In [ ]:
colineales = []


for c in df._get_numeric_data():
    
    for i in range(len(df.corr())):
        
        if abs(df.corr()[c][i])>0.9 and abs(df.corr()[c][i])<1:
            
            colineales.append(c)
            
colineales

In [ ]:
colineales = list(set(colineales))

colineales

In [ ]:
df.drop(columns=colineales, inplace=True)

In [ ]:
dim_original, df.shape

In [ ]:
df.info(memory_usage='deep')

### Limpieza columna `trany`

In [ ]:
df.trany.value_counts()

In [ ]:
import re   # regex

int(re.findall('\d+', 'Automatic (AM6)')[0])

In [ ]:
def limpiar_trany(x):
    
    num = re.findall('\d+', x)
    
    if num:  # si hay numero, la lista no esta vacia
        
        return int(num[0])
    
    else:
        
        return 0

In [ ]:
limpiar_trany('Automatic (AM6)')

In [ ]:
df['speed'] = df.trany.apply(limpiar_trany)

df.head()

In [ ]:
df.trany.value_counts()

In [ ]:
def limpiar_auto(x):
    
    x = x.lower()
    
    if 'auto' in x:
        return 'automatic'
    
    elif 'man' in x:
        return 'manual'
    
    else:
        return 'unknown'

In [ ]:
df.trany = df.trany.apply(limpiar_auto)

df.head()

In [ ]:
df.trany.value_counts()

In [ ]:
df[df.trany=='unknown'].index

In [ ]:
df.drop(index=df[df.trany=='unknown'].index, inplace=True)

In [ ]:
df.trany.value_counts()

In [ ]:
df.shape, dim_original

In [ ]:
df.drop_duplicates().shape==df.shape

## DataFrame Final

In [ ]:
df.head()

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df.to_excel('../data/vehiculos_limpio.xls', index=False)

In [ ]:
%pip install xlwt
%pip install xlrd

In [ ]:
df.to_excel('../data/vehiculos_limpio.xlsx', index=False)

In [ ]:
df.to_csv('../data/vehiculos_limpio.csv', index=False)

**Cuidado de no eliminarlo todo....**